# Overlay design

The H14 walkthrough, generalised: any signals, any regime model, weights per state. An overlay earns its place only
if it beats its **static blend** (the same signals at their average weights) and its **exposure-matched static**
(the overlay's exposure path with the static blend's direction): otherwise the regime adds turnover, not information.

Every weight set tried below is a configuration. Exploring is fine; report how many were tried when you
pre-register (`lab.exploration_count()`).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.lab import Lab
from quantlab.signals import library as lib
from quantlab.reporting import charts

charts.apply_style()
pd.set_option("display.width", 200)

from quantlab.regimes.models import FeatureBlendRegime, PercentileRegime, VolatilityRegime, labels
from quantlab.analytics.conditional import by_regime

lab = Lab(universe="equity_index_core", start="1999")
reversion = lab.signal(lib.zscore_reversion(40), name="reversion 40")

## Signals, regime and state weights

In [ ]:
regime = PercentileRegime("er", state_names=("RANGE", "TREND"))
state_weights = {"TREND": {"trend": 1.0, "reversion 40": 0.0},
                 "RANGE": {"trend": 0.3, "reversion 40": 0.7}}
design = lab.combine("trend", reversion, overlay=regime, state_weights=state_weights)

## A few alternatives (each one is a configuration tried)

In [ ]:
alternatives = {
    "range: reversion only": {"TREND": {"trend": 1.0}, "RANGE": {"reversion 40": 1.0}},
    "range: flat": {"TREND": {"trend": 1.0}, "RANGE": {}},
    "trend everywhere, reversion in range": {"TREND": {"trend": 1.0}, "RANGE": {"trend": 1.0, "reversion 40": 0.5}},
}
rows = {}
for name, weights in alternatives.items():
    out = lab.combine("trend", reversion, overlay=regime, state_weights=weights, show=False)
    rows[name] = out["table"].loc[["overlay", "static blend", "exposure-matched static"], "Sharpe"]
pd.DataFrame(rows).T.round(3)

## Where the overlay earns (descriptive: labels and P&L overlap in time)

In [ ]:
market = lab.instruments[0]
state = labels(regime(lab.panel.bars(market)))
results = design["results"]
by_regime({name: results[name].net[market] for name in ["overlay", "static blend", "trend alone"]}, state).round(2)